<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/topkvsinf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U \
    langchain \
    langchain-community \
    langchain-huggingface \
    sentence-transformers \
    faiss-cpu \
    transformers \
    accelerate

In [ ]:
!pip install langchain_openai

In [ ]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

In [ ]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    set_seed,
)

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFacePipeline

ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
documents = [
    Document(
        page_content=(
            "KV cache stores previously computed key and value tensors "
            "so they do not need to be recalculated during every decoding step."
        ),
        metadata={"topic": "kv-cache"},
    ),
    Document(
        page_content=(
            "Paged attention manages KV cache memory using fixed-size blocks "
            "that can be allocated dynamically to requests."
        ),
        metadata={"topic": "paged-attention"},
    ),
    Document(
        page_content=(
            "Continuous batching allows new requests to enter an active batch "
            "while other requests are generating tokens."
        ),
        metadata={"topic": "continuous-batching"},
    ),
    Document(
        page_content=(
            "Temperature changes the next-token probability distribution "
            "during language model generation."
        ),
        metadata={"topic": "sampling"},
    ),
    Document(
        page_content=(
            "Top-k sampling keeps only the k most probable next-token "
            "candidates before sampling one token."
        ),
        metadata={"topic": "sampling"},
    ),
]

In [ ]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_API_KEY
)

In [ ]:
vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embeddings,
)

print("Vector Store Created.")

In [ ]:
retrieval_top_k=3

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": retrieval_top_k,
    },
)

In [ ]:
query = "How does an inference engine manage KV Cache memory?"

retrieved_documents = retriever.invoke(query)

print(f"Retrieved {len(retrieved_documents)} documents\n")

for rank, document in enumerate(retrieved_documents, start=1):
  print(f"Rank {rank}")
  print(document.page_content)
  print("Metadata:", document.metadata)
  print("-" * 80)

In [ ]:
MODEL_NAME="Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
)

print("Model loaded.")

In [ ]:
sampling_top_k = 3

set_seed(42)

generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,


    do_sample=True,
    temperature=0.7,
    top_k=sampling_top_k,
    top_p=1.0,

    max_new_tokens=150,
    repetition_penalty=1.05,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)

In [ ]:
test_prompt = """
Explain top-k sampling in LLM inference in two sentences.
"""

result = generator(test_prompt)

print(result[0]["generated_text"])

In [ ]:
llm = HuggingFacePipeline(
    pipeline=generator
)

In [ ]:
prompt = ChatPromptTemplate.from_template(
"""
Answer the question using only the provided context.

If the answer is not present in the context, say:
"I do not have enough information."

Context:
{context}

Question:
{question}

Answer:
"""
)

In [ ]:
def format_documents(documents):
      return "\n\n".join(
        f"Document {index}:\n{document.page_content}"
        for index, document in enumerate(documents, start=1)
    )

In [ ]:
rag_chain = (
    {
        "context": retriever | format_documents,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
query = "How does paged attention manage KV cache memory?"

answer = rag_chain.invoke(query)

print("Retrieval top-k:", retrieval_top_k)
print("Inference top-k:", sampling_top_k)
print("\nAnswer:\n")
print(answer)

In [ ]:
import torch

sampling_top_k = 3

test_prompt = "The capital of France is"

inputs = tokenizer(
    test_prompt,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model(**inputs)


next_token_logits = outputs.logits[0, -1, :]


next_token_probabilities = torch.softmax(
    next_token_logits,
    dim=-1,
)


top_probabilities, top_token_ids = torch.topk(
    next_token_probabilities,
    k=sampling_top_k,
)

print("=" * 70)
print("INFERENCE TOP-K")
print("=" * 70)
print(f"sampling_top_k = {sampling_top_k}")
print(
    f"Meaning: keep the {sampling_top_k} most probable "
    "next-token candidates.\n"
)

for rank, (token_id, probability) in enumerate(
    zip(top_token_ids, top_probabilities),
    start=1,
):
    token = tokenizer.decode([token_id.item()])

    print(
        f"Candidate {rank}: "
        f"token={repr(token)}, "
        f"token_id={token_id.item()}, "
        f"probability={probability.item():.4f}"
    )

In [ ]:
query = "How does paged attention manage KV cache memory?"

retrieval_top_k = 3

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": retrieval_top_k},
)

retrieved_documents = retriever.invoke(query)


print("=" * 70)
print("RETRIEVAL TOP-K")
print("=" * 70)
print(f"retrieval_top_k = {retrieval_top_k}")
print(f"Meaning: retrieve {retrieval_top_k} documents from the vector database.\n")

for rank, document in enumerate(retrieved_documents, start=1):
    print(f"Retrieved document {rank}")
    print(document.page_content)
    print("-" * 70)

In [ ]:
filtered_probabilities = top_probabilities / top_probabilities.sum()